In [112]:
import pickle as pl
from sklearn.metrics import PrecisionRecallDisplay
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.metrics import roc_curve, auc
from ordered_set import OrderedSet
import os


In [113]:
with open('/Users/adityagoyal/Desktop/Research - yin li/baseline/results5.pkl','rb') as f:
    loaded_data = pl.load(f)

In [114]:
loaded_data[(0,7)][0]['y_scores'][0][0]
loaded_data[(0,7)][0]['num_features']

[47, 42, 37, 32, 27, 22, 17, 12, 7, 5]

In [115]:
print(len(loaded_data[(0,7)][0]['y_scores'][0][0]))
print(len(loaded_data[(0,7)][0]['y_val'][0][0]))

14032
14032


In [116]:
loaded_data[(0,7)][0]['roc_auc_score'][0]
precision, recall,thresholds = precision_recall_curve(loaded_data[(0,7)][0]['y_val'][0][0],loaded_data[(0,7)][0]['y_scores'][0][0])

In [ ]:
def plot_mean_roc(y_true_folds, y_score_folds, df_filepath):
    fpr_grid = np.linspace(0, 1, 1000)
    tpr_interp = []
    aucs = []

    for y_true, y_score in zip(y_true_folds, y_score_folds):
        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc = auc(fpr, tpr)
        aucs.append(roc_auc)
        # Interpolate TPR at common FPR points
        tpr_interp.append(np.interp(fpr_grid, fpr, tpr))

    mean_tpr = np.mean(tpr_interp, axis=0)
    std_tpr = np.std(tpr_interp, axis=0)
    mean_auc = np.mean(aucs)

    plt.figure(figsize=(6, 6))
    plt.plot(fpr_grid, mean_tpr, label=f"ROC Curve (AUC = {mean_auc:.3f})")
    plt.fill_between(fpr_grid, mean_tpr - std_tpr, mean_tpr + std_tpr, alpha=0.2)
    plt.plot([0, 1], [0, 1], 'k--',linewidth=1,color='k',alpha=0.5)
    plt.xlabel("False Positive Rate",fontsize=12)
    plt.ylabel("True Positive Rate",fontsize=12)
    plt.title("ROC Curve (10-fold CV)",fontsize=13)
    plt.legend()
    plt.grid(True)
    # plt.text(0.5, 0.4, f'No of features: {num_features}', fontsize=12, color='red')
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.tick_params(axis='both', which='major', labelsize=12)
    plt.grid(False)
    plt.savefig(f"{df_filepath}/ROC curve.png", dpi=300, bbox_inches="tight")
    plt.close()

In [ ]:
# Suppose you have lists of predictions and true labels for each fold:
# y_true_folds = [y_true_fold1, y_true_fold2, ..., y_true_fold10]
# y_pred_prob_folds = [y_prob_fold1, y_prob_fold2, ..., y_prob_fold10]
 
# Example structure:
# y_true_folds = [...]
# y_pred_prob_folds = [...]
 
# Define a common recall grid for interpolation
def plot_pr(y_true_folds, y_pred_prob_folds,df_filepath):
    recall_grid = np.linspace(0, 1, 1000)
    
    # Store interpolated precision values
    precision_interp = []
    average_precisions = []
    
    for y_true, y_pred_prob in zip(y_true_folds,y_pred_prob_folds):
        precision, recall, _ = precision_recall_curve(y_true, y_pred_prob)
        ap = average_precision_score(y_true, y_pred_prob)
        average_precisions.append(ap)
    
        # Interpolate precision at common recall points
        precision_interp.append(np.interp(recall_grid, recall[::-1], precision[::-1]))
    
    # Compute mean and std of precision at each recall point
    mean_precision = np.mean(precision_interp, axis=0)
    std_precision = np.std(precision_interp, axis=0)
    mean_ap = np.mean(average_precisions)
    
    # Plot mean PR curve with shaded variance
    plt.figure(figsize=(6, 6))
    plt.plot(recall_grid, mean_precision, label=f"PR Curve (AUPRC = {mean_ap:.3f})")
    plt.fill_between(recall_grid,
                    mean_precision - std_precision,
                    mean_precision + std_precision,
                    alpha=0.2)
    plt.xlabel("Recall",fontsize=12)
    plt.ylabel("Precision",fontsize=12)
    plt.title("Precision/Recall Curve (10-fold CV)",fontsize=13)
    plt.legend(fontsize=11)
    plt.grid(True)
    # plt.text(0.5, 0.4, f'No of features: {num_features}', fontsize=12, color='red')
    plt.xlim(0,1)
    plt.ylim(0,1)
    plt.tick_params(axis='both', which='major', labelsize=12)
    plt.tight_layout()
    plt.grid(False)
    plt.savefig(f"{df_filepath}/PR curve.png", dpi=300, bbox_inches="tight")
    plt.close()
    return mean_precision,recall_grid

In [119]:
# go through all the candiate features, and take the max roc-auc of it , find the setting for a dataframe and date

In [120]:
# do the pr tables for that setting 
# pr curve 
# roc-auc curve

In [121]:
dfs_dict = { 0: 'df_hr_spo2' , 1: 'df_hr_spo2_demographics' }
days = OrderedSet(( (0,1),(0,7),(3,4),(7,8)))

In [122]:
def store_plots(results_dir,loaded_data):
    """
    Saves plots and tables of results for each day and feature group.

    Args:
        results_dir (str): Directory to save results.
        loaded_data (dict): Results data loaded from pickle.
    """
    output_dir = results_dir

    # Create the directory if it doesn't exist
    if not os.path.isdir(output_dir):
        os.makedirs(output_dir)
        
    for day in days: 
        # create dir for day
        
        output_dir = results_dir
        filename = f"Day: {day}"
        day_filepath = os.path.join(output_dir, filename)
        if not os.path.isdir(day_filepath):
            os.makedirs(day_filepath)
        
        
        for dataframe_index in range(len(dfs_dict)):
            
            # creating the plots
            
            # create dir for dataframe_index..this is the last dir
            output_dir =  day_filepath
            filename = f"{dfs_dict[dataframe_index]}"
            df_filepath = os.path.join(output_dir, filename)
            if not os.path.isdir(df_filepath):
                os.makedirs(df_filepath)
            
            # get the stats from the result
            result = loaded_data[day][dataframe_index]
            roc_auc_scores = result['roc_auc_score']
            aurpc_scores = result['aurpc_score']
            plot_index = np.argmax(roc_auc_scores)
            num_features = result['num_features'][plot_index]
            y_scores = result['y_scores'][plot_index]
            y_val = result['y_val'][plot_index]
            
            # plotting the PR curve
            precision,recall = plot_pr(y_val,y_scores,df_filepath,num_features)
            
            # plotting the ROC-AUC curve
            plot_mean_roc(y_val,y_scores, df_filepath, num_features)
            
            # plotting the table
            fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(5, 5),squeeze=False)

            columns = ['recall','precision']
                
            ax=axes[0][0]
            ax.axis('off')
            
            values = [0.5,0.55,0.6,0.65,0.7,0.75,0.8,0.85,0.9,0.95,0.99]
            data = []
            
            for value in values:
                recall_values = abs(recall - value)
                index = int(np.argmin(recall_values))
                
                # recall = recall_values[index]
                # if(recall>0.01):
                #     print(recall)
                #     print(f"True Day: {day}  df_index: {dataframe_index}  plot_index: {plot_index}")
                #     print(len(np.unique(loaded_data[day][dataframe_index]['y_scores'][plot_index][0])))
                #     print('\n')
                    
                precision_val =  precision[index]
                data.append([value,precision_val])
                
            table = ax.table(cellText=data, colLabels=columns, loc='center')
            ax.text(0.5, 0.07,  f'No of features: {num_features}', fontsize=12, color='red')
            ax.text(0.5, 0.14,  f'AURPC score: {int(aurpc_scores[plot_index]*1000) /1000}', fontsize=12, color='red')
            
            plt.tight_layout()
            plt.savefig(f"{df_filepath}/tables.png", dpi=300, bbox_inches="tight")
            plt.close()


In [123]:
# title (10-fold CV) 13
# label : 12 
# plt legend: 11

In [124]:
store_plots('/Users/adityagoyal/Desktop/Research - yin li/baseline/results8',loaded_data)

/var/folders/tg/dclk92x95vn7bc58nqqc369w0000gn/T/ipykernel_46617/2304298758.py:20: UserWarning: color is redundantly defined by the 'color' keyword argument and the fmt string "k--" (-> color='k'). The keyword argument will take precedence.
  plt.plot([0, 1], [0, 1], 'k--',linewidth=1,color='k',alpha=0.5)
/var/folders/tg/dclk92x95vn7bc58nqqc369w0000gn/T/ipykernel_46617/2304298758.py:20: UserWarning: color is redundantly defined by the 'color' keyword argument and the fmt string "k--" (-> color='k'). The keyword argument will take precedence.
  plt.plot([0, 1], [0, 1], 'k--',linewidth=1,color='k',alpha=0.5)
/var/folders/tg/dclk92x95vn7bc58nqqc369w0000gn/T/ipykernel_46617/2304298758.py:20: UserWarning: color is redundantly defined by the 'color' keyword argument and the fmt string "k--" (-> color='k'). The keyword argument will take precedence.
  plt.plot([0, 1], [0, 1], 'k--',linewidth=1,color='k',alpha=0.5)
/var/folders/tg/dclk92x95vn7bc58nqqc369w0000gn/T/ipykernel_46617/2304298758.py:

In [85]:
# x_recalls = np.array(result['recalls_arr_mean'])
# x_fprs = np.array(result['false_positive_rates_arr_mean'])
# x_recall_stds = np.array(result['recall_arr_std'])

# fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(5, 5),squeeze=False)

# ax=axes[0][0]

# x_recall = x_recalls[plot_index]
# x_fpr = x_fprs[plot_index]

# x_recall_std = x_recall_stds[plot_index]
# roc_auc_score = roc_auc_scores[plot_index]

# lower = np.maximum(x_recall - x_recall_std, 0.0).reshape(-1)
# upper = np.minimum(x_recall + x_recall_std, 1.0).reshape(-1)
# ax.plot(x_fpr,x_recall, color="blue")
# ax.set_title("ROC-AUC curve")
# ax.set_xlim(0,1)
# ax.set_ylim(0,1)
# ax.fill_between(x_fpr.reshape(-1), lower, upper, alpha=0.3,color='0.8')
# ax.text(0.5, 0.5, f'AUC-ROC score: {int(roc_auc_score*1000)/1000}', fontsize=12, color='red')
# ax.text(0.5, 0.4,  f'No of features: {num_features}', fontsize=12, color='red')
# ax.set_xlabel("FPR")
# ax.set_ylabel("Recall")
    
# plt.tight_layout()
# plt.savefig(f"{df_filepath}/ROC-AUC curve.png", dpi=300, bbox_inches="tight")
# plt.close()

In [58]:
# loaded_data.keys()
# for day in loaded_data:
#     for dataframe_index in dfs_dict:
#         result = loaded_data[day][dataframe_index]
#         roc_auc_scores = result['roc_auc_score']
#         max_index = np.argmax(roc_auc_scores)
        
#         y_scores = result['y_scores'][max_index]
#         y_val = result['y_val'][max_index]
#         num_features = result['num_features'][max_index]
#         plot_pr(y_val, y_scores)
        